# **Business Understanding**

The primary objective of this analysis is to understand customer purchasing behavior and identify opportunities to improve customer retention and revenue. Through customer segmentation and data analysis, this project aims to answer key business questions and provide actionable recommendations.

## **Business Objectives**

• Identify high-value and low-value customer segments. 

• Analyze customer purchasing behavior and buying patterns.

• Identify customers who are at risk of churn.

• Recommend strategies to improve customer retention and revenue growth.

• Generate data-driven business insights to support decision making.

### **Business Questions**
• Who are the most valuable customers?

• Which customer segments contribute the highest revenue?

• Which customers are at risk of churn?

• What purchasing patterns can be observed?

• What recommendations can improve customer retention?

# **Dataset Discription**

The analysis is performed using the **Olist Brazilian E-Commerce Dataset**, a publicly available dataset from Kaggle that contains transactional data from a Brazilian e-commerce marketplace. The dataset follows a relational database structure, where multiple tables are connected through customer and order identifiers. The database schema is provided in `/images/database_schema.png`.

Dataset link : https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce

For this project, only the tables required to answer the business objectives are used. The primary datasets include customer information, order details, payment records, and order item information. Additional tables such as product information and customer reviews may be incorporated later to perform category-level analysis and customer satisfaction analysis.

The relational structure of the dataset enables comprehensive customer-level analysis by combining information across multiple tables. This makes it suitable for customer segmentation, purchasing behavior analysis, revenue analysis, retention analysis, and generating business recommendations.



In [32]:
import warnings 
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import chi2_contingency
from scipy.stats import pearsonr

In [33]:
orders = pd.read_csv("../data/olist_orders_dataset.csv")

customers = pd.read_csv("../data/olist_customers_dataset.csv")

payments = pd.read_csv("../data/olist_order_payments_dataset.csv")

items = pd.read_csv("../data/olist_order_items_dataset.csv")

In [34]:
def inspect_df(df, name):
    print("=" * 60)
    print(f"{name.upper()} DATASET")
    print("=" * 60)

    print("\nShape:")
    print(df.shape)

    print("\nSample:")
    display(df.sample(5))

    print("\nInfo:")
    df.info()

    print("\nDescribe:")
    display(df.describe(include="all"))

    print("\nMissing Values:")
    display(df.isnull().sum())

    print("\nDuplicates:")
    print(df.duplicated().sum())

In [35]:
datasets = {
    "Orders": orders,
    "Customers": customers,
    "Payments": payments,
    "Order Items": items
}

for name, df in datasets.items():
    inspect_df(df, name)

ORDERS DATASET

Shape:
(99441, 8)

Sample:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
25107,ae0c703b9ca19474ade7bec18099ad7f,472a39839e3ad5a790edb3721639caa4,delivered,2017-08-15 20:28:18,2017-08-17 03:06:09,2017-08-17 16:08:38,2017-08-30 18:42:06,2017-09-12 00:00:00
72458,863a5aae025377c66a973980616bb64f,af2e52573cb16f3a14e6306913817975,delivered,2017-12-07 10:59:43,2017-12-08 02:36:54,2017-12-11 19:37:02,2017-12-18 17:18:41,2018-01-04 00:00:00
30599,d07d8db4850966f820d3775360ccd6b8,20789090e052be519b3e8d97813bbfa2,delivered,2017-07-25 12:06:38,2017-07-26 02:25:40,2017-07-26 19:21:23,2017-08-07 18:34:30,2017-08-18 00:00:00
23555,5fd0d1844c46d95e756c563bf148d01d,4bff33bda9d28aa45c0a3883999849bb,delivered,2018-03-15 13:35:00,2018-03-16 03:45:21,2018-03-17 00:26:43,2018-03-22 00:22:27,2018-04-03 00:00:00
77463,6adf72edbe32f7b03c39edd4b0b97fe5,95ae7c675a8591a773af47661e236ea3,delivered,2018-06-30 22:59:52,2018-06-30 23:15:10,2018-07-02 13:59:00,2018-07-06 17:21:19,2018-07-25 00:00:00



Info:
<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB

Describe:


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2018-03-31 15:08:21,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522



Missing Values:


...


Info:
<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype  
---  ------               --------------   -----  
 0   order_id             112650 non-null  str    
 1   order_item_id        112650 non-null  int64  
 2   product_id           112650 non-null  str    
 3   seller_id            112650 non-null  str    
 4   shipping_limit_date  112650 non-null  str    
 5   price                112650 non-null  float64
 6   freight_value        112650 non-null  float64
dtypes: float64(2), int64(1), str(4)
memory usage: 6.0 MB

Describe:


,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
count,112650,112650.000000,112650,112650,112650,112650.000000,112650.000000
unique,98666,NaN,32951,3095,93318,NaN,NaN
top,8272b63d03f5f79c56e9e4120aec44ef,NaN,aca2eb7d00ea1a7b8ebd4e68314663af,6560211a19b47992c3666cc44a7e94c0,2018-03-01 02:50:48,NaN,NaN
freq,21,NaN,527,2033,21,NaN,NaN
mean,NaN,1.197834,NaN,NaN,NaN,120.653739,19.990320
std,NaN,0.705124,NaN,NaN,NaN,183.633928,15.806405
min,NaN,1.000000,NaN,NaN,NaN,0.850000,0.000000
25%,NaN,1.000000,NaN,NaN,NaN,39.900000,13.080000
50%,NaN,1.000000,NaN,NaN,NaN,74.990000,16.260000
75%,NaN,1.000000,NaN,NaN,NaN,134.900000,21.150000



Missing Values:


order_id               0
order_item_id          0
product_id             0
seller_id              0
shipping_limit_date    0
price                  0
freight_value          0
dtype: int64


Duplicates:
0


### Orders Dataset
- Dataset contains 99,441 orders and 8 columns.
- Several date-related columns are stored as object and will be converted to datetime for time-based analysis.
- Missing values are present in order_approved_at, order_delivered_carrier_date, and order_delivered_customer_date. These will be investigated before any cleaning decisions are made.
- No duplicate records were found.

### Customers Dataset
- Dataset contains 99,441 customer records.
- All columns have appropriate data types.
- No missing values or duplicate records were found.
- This table will be used to identify unique customers and perform location-based analysis.

### Payments Dataset
- Dataset contains 103,886 payment records, indicating that some orders have multiple payment entries.
- No missing values or duplicate records were found.
- Data types are appropriate for further monetary analysis.

### Order Items Dataset
- Dataset contains 112,650 order item records, indicating that an order may contain multiple products.
- shipping_limit_date is stored as a string and will be converted to datetime.
- No missing values or duplicate records were found.

In [36]:
# verification

orders['order_status'].value_counts()

order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

In [37]:
orders.groupby("order_status")[
    [
        "order_approved_at",
        "order_delivered_carrier_date",
        "order_delivered_customer_date",
    ]
].apply(lambda x: x.isnull().sum())

,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date
order_status,,,
approved,0,2,2
canceled,141,550,619
created,5,5,5
delivered,14,2,8
invoiced,0,314,314
processing,0,301,301
shipped,0,0,1107
unavailable,0,609,609


In [38]:
orders.isnull().mean()*100

order_id                         0.000000
customer_id                      0.000000
order_status                     0.000000
order_purchase_timestamp         0.000000
order_approved_at                0.160899
order_delivered_carrier_date     1.793023
order_delivered_customer_date    2.981668
order_estimated_delivery_date    0.000000
dtype: float64

### **3. Data Cleaning Strategy**

- Datetime columns will be converted using pd.to_datetime().

- Missing delivery-related timestamps are expected for orders that were canceled, unavailable, or are still in progress.

- A very small number of delivered orders also contain missing timestamps, which may indicate data quality issues.

- Since delivery timestamps are not required for RFM analysis, these missing values will not be imputed at this stage.

In [43]:
datetime_cols = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

for col in datetime_cols:
    orders[col] = pd.to_datetime(orders[col], errors="coerce")

In [44]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


In [45]:
items["shipping_limit_date"] = pd.to_datetime(items["shipping_limit_date"], errors="coerce")

In [46]:
items.info()

<class 'pandas.DataFrame'>
RangeIndex: 112650 entries, 0 to 112649
Data columns (total 7 columns):
 #   Column               Non-Null Count   Dtype         
---  ------               --------------   -----         
 0   order_id             112650 non-null  str           
 1   order_item_id        112650 non-null  int64         
 2   product_id           112650 non-null  str           
 3   seller_id            112650 non-null  str           
 4   shipping_limit_date  112650 non-null  datetime64[us]
 5   price                112650 non-null  float64       
 6   freight_value        112650 non-null  float64       
dtypes: datetime64[us](1), float64(2), int64(1), str(3)
memory usage: 6.0 MB


In [47]:
payments.head()

,order_id,payment_sequential,payment_type,payment_installments,payment_value
0,b81ef226f3fe1789b1e8b2acac839d17,1,credit_card,8,99.33
1,a9810da82917af2d9aefd1278f1dcfa0,1,credit_card,1,24.39
2,25e8ea4e93396b6fa0d3dd708e76c1bd,1,credit_card,1,65.71
3,ba78997921bbcdc1373bb41e913ab953,1,credit_card,8,107.78
4,42fdf880ba16b47b59251dd489d4441a,1,credit_card,2,128.45


### Joins

In [72]:
payment_summary = payments.groupby("order_id").agg(
    total_payment=("payment_value", sum)
).reset_index()

In [73]:
items_summary = items.groupby("order_id").agg(
    total_price = ("price", "sum"),
    total_freight = ("freight_value", "sum"),
    total_items = ("order_item_id", "count")
).reset_index()

In [74]:
master_df = (
    orders
    .merge(customers, on="customer_id", how="left")
    .merge(payment_summary, on="order_id", how="left")
    .merge(items_summary, on="order_id", how="left")
)

In [75]:
master_df.shape

(99441, 16)

In [76]:
master_df.head()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state,total_payment,total_price,total_freight,total_items
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18,7c396fd4830fd04220f754e42b4e5bff,3149,sao paulo,SP,38.71,29.99,8.72,1.0
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13,af07308b275d755c9edb36a90c618231,47813,barreiras,BA,141.46,118.70,22.76,1.0
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04,3a653a41f6f9fc3d2a113cf8398680e8,75265,vianopolis,GO,179.12,159.90,19.22,1.0
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15,7c142cf63193a1473d2e66489a9ae977,59296,sao goncalo do amarante,RN,72.20,45.00,27.20,1.0
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26,72632f0f9dd73dfee390c9b22eb56dd6,9195,santo andre,SP,28.62,19.90,8.72,1.0


In [77]:
master_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 16 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
 8   customer_unique_id             99441 non-null  str           
 9   customer_zip_code_prefix       99441 non-null  int64         
 10  customer_city                  99441 non-null  str           
 11  customer_state            

### **4. RFM Feature Engineering**

In [102]:
rfm_df = master_df[master_df['order_status'] == "delivered"].copy()

In [103]:
rfm_df.shape

(96478, 16)

In [104]:
rfm = rfm_df.groupby('customer_unique_id').agg(
    LastPurchase=("order_purchase_timestamp", "max"),
    Frequency=("order_id", "nunique"),
    Monetary=("total_payment", "sum"),
)

In [105]:
refrence_date = rfm['LastPurchase'].max() + pd.Timedelta(days=1)

rfm["Recency"] = (refrence_date - rfm['LastPurchase']).dt.days

In [106]:
rfm.drop(columns={"LastPurchase"}, inplace=True)
rfm = rfm[["Recency", "Frequency", "Monetary"]]

In [108]:
rfm.describe()

,Recency,Frequency,Monetary
count,93358.000000,93358.000000,93358.000000
mean,237.941773,1.033420,165.197003
std,152.591453,0.209097,226.314012
min,1.000000,1.000000,0.000000
25%,114.000000,1.000000,63.052500
50%,219.000000,1.000000,107.780000
75%,346.000000,1.000000,182.557500
max,714.000000,15.000000,13664.080000


In [110]:
rfm["R_Score"] = pd.qcut(
    rfm["Recency"],
    q=4,
    labels=[4, 3, 2, 1]
)

I used pd.qcut() because RFM scoring is percentile-based. It divides customers into equal-sized groups (quantiles), allowing fair comparison of customer behavior even when the data distribution is skewed.

In [116]:
rfm["Frequency"].value_counts().head(10)

Frequency
1     90557
2      2573
3       181
4        28
5         9
6         5
7         3
9         1
15        1
Name: count, dtype: int64

The frequency distribution was highly skewed, with most customers placing only one order. Quantile binning could not create meaningful quartiles, so I used business-rule-based scoring to create interpretable customer segments.

In [117]:
def frequency_score(x):
    if x == 1:
        return 1
    elif x == 2:
        return 2
    elif 3 <= x <= 4:
        return 3
    else:
        return 4

rfm["F_Score"] = rfm["Frequency"].apply(frequency_score)

In [118]:
rfm.head()

,Recency,Frequency,Monetary,R_Score,M_Score,F_Score
customer_unique_id,,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,3,1
0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,3,1,1
0000f46a3911fa3c0805444483337064,537,1,86.22,1,2,1
0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1
0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,4,1


In [112]:
rfm["M_Score"] = pd.qcut(
    rfm["Monetary"],
    q=4,
    labels=[1,2,3,4]
)

In [119]:
rfm["rfm_score"] = (
    rfm['R_Score'].astype(str)+
    rfm['F_Score'].astype(str)+
    rfm['M_Score'].astype(str)
)

In [120]:
rfm.head()

,Recency,Frequency,Monetary,R_Score,M_Score,F_Score,rfm_score
customer_unique_id,,,,,,,
0000366f3b9a7992bf8c76cfdf3221e2,112,1,141.90,4,3,1,413
0000b849f77a49e4a4ce2b2a4ca5be3f,115,1,27.19,3,1,1,311
0000f46a3911fa3c0805444483337064,537,1,86.22,1,2,1,112
0000f6ccb0745a6a4b88665a16c9f078,321,1,43.62,2,1,1,211
0004aac84e0df4da2b147fca70cf8255,288,1,196.89,2,4,1,214


Frequency scoring was performed using business-defined thresholds instead of quantile binning because the frequency distribution was highly skewed. More than 90,000 customers had only one purchase, making quartile-based segmentation unsuitable.

In [123]:
def segment_customer(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]

    if r == 4 and f >= 3 and m >= 3:
        return "Champions"

    elif r >= 3 and f >= 2:
        return "Loyal Customers"

    elif r == 4 and f == 1:
        return "Potential Loyalists"

    elif r == 2 and f == 1:
        return "Needs Attention"

    elif r == 1 and f >= 2:
        return "At Risk"

    else:
        return "Lost"

rfm["Segment"] = rfm.apply(segment_customer, axis=1)

In [124]:
rfm["Segment"].value_counts()

Segment
Lost                   46090
Potential Loyalists    22613
Needs Attention        22527
Loyal Customers         1474
At Risk                  579
Champions                 75
Name: count, dtype: int64

In [125]:
rfm.groupby("Segment").agg({
    "Recency": "mean",
    "Frequency": "mean",
    "Monetary": "mean"
}).round(2)

,Recency,Frequency,Monetary
Segment,,,
At Risk,439.65,2.08,291.54
Champions,57.77,3.61,627.89
Lost,308.88,1.02,162.02
Loyal Customers,115.77,2.07,298.80
Needs Attention,277.33,1.00,158.73
Potential Loyalists,57.50,1.00,164.64


If Potential Loyalists can be encouraged to make repeat purchases, they have a high chance of becoming Champions.